# Módulo 2 — Análise de Qualidade do Cadastro

Este notebook constrói uma análise completa de qualidade do cadastro de consumidores, cobrindo as quatro dimensões fundamentais: **Completude**, **Consistência**, **Unicidade** e **Validade**.

## O que vamos ver

1. Completude — % de campos preenchidos por coluna e por registro
2. Consistência — relações entre campos (classe vs. modalidade tarifária)
3. Detecção de outliers no consumo
4. Relatório consolidado de problemas

---

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import date

# Carregar dados originais
df = pd.read_csv("dados/consumidores_simulado.csv")
df["dat_ultima_leitura"] = pd.to_datetime(df["dat_ultima_leitura"], errors="coerce")

print(f"Total de registros: {len(df)}")
df.head(3)

## 1. Completude — Percentual de Preenchimento

In [ ]:
def analisar_completude(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula a completude (% de preenchimento) por coluna.
    
    Returns:
        DataFrame com: coluna, total, preenchidos, nulos, pct_preenchimento, status
    """
    resultado = []
    
    for col in df.columns:
        total = len(df)
        nulos = df[col].isnull().sum()
        
        # Também contar strings vazias como nulo
        if df[col].dtype == object:
            nulos += (df[col].fillna("").str.strip() == "").sum()
            nulos = min(nulos, total)  # não pode ultrapassar o total
        
        preenchidos = total - nulos
        pct = round(preenchidos / total * 100, 1)
        
        if pct >= 95:
            status = "OK"
        elif pct >= 80:
            status = "ATENÇÃO"
        else:
            status = "CRÍTICO"
        
        resultado.append({
            "coluna": col,
            "total": total,
            "preenchidos": preenchidos,
            "nulos": nulos,
            "pct_preenchimento": pct,
            "status": status
        })
    
    return pd.DataFrame(resultado).sort_values("pct_preenchimento")


completude = analisar_completude(df)
print("Análise de Completude por Coluna:")
print()
print(f"{'Coluna':<35} | {'Preench%':>9} | {'Nulos':>6} | {'Status'}")
print("-" * 65)
for _, row in completude.iterrows():
    emoji = "" if row["status"] == "OK" else "" if row["status"] == "ATENÇÃO" else ""
    barra = "█" * int(row["pct_preenchimento"] / 5)
    print(f"{row['coluna']:<35} | {row['pct_preenchimento']:>8.1f}% | {row['nulos']:>6} | {row['status']}")

In [ ]:
# Score de completude por registro
campos_obrigatorios = [
    "nom_consumidor", "cpf_cnpj", "cod_modalidade_tarifaria",
    "cod_classe_consumo", "nom_municipio", "cod_uf",
    "num_cep", "num_medidor", "dat_ultima_leitura"
]

df["score_completude"] = df[campos_obrigatorios].notna().sum(axis=1) / len(campos_obrigatorios)

print(f"Score médio de completude: {df['score_completude'].mean():.1%}")
print(f"Registros com completude 100%: {(df['score_completude'] == 1.0).sum()}")
print(f"Registros com completude < 80%: {(df['score_completude'] < 0.8).sum()}")

## 2. Consistência — Relações Entre Campos

In [ ]:
# Mapeamento esperado: classe de consumo → modalidades tarifárias permitidas
# Conforme PRODIST/ANEEL
MODALIDADES_POR_CLASSE = {
    "RESIDENCIAL":      ["B1"],
    "RURAL":            ["B2"],
    "COMERCIAL":        ["B3", "A4", "A3a", "A3", "A2", "A1"],
    "INDUSTRIAL":       ["B3", "A4", "A3a", "A3", "A2", "A1", "AS"],
    "PODER PUBLICO":    ["B3", "A4", "A3a", "A3", "A2", "A1"],
    "SERVICO PUBLICO":  ["B3", "A4", "A3a", "A3", "A2", "A1"],
    "ILUMINACAO PUBLICA": ["B4"],
}

def verificar_consistencia_tarifaria(row) -> str:
    """
    Verifica se a modalidade tarifária é consistente com a classe de consumo.
    """
    classe = str(row["cod_classe_consumo"]).upper().strip() if pd.notna(row["cod_classe_consumo"]) else None
    modalidade = str(row["cod_modalidade_tarifaria"]).upper().strip() if pd.notna(row["cod_modalidade_tarifaria"]) else None
    
    if classe is None or modalidade is None:
        return "DADO_AUSENTE"
    
    if classe not in MODALIDADES_POR_CLASSE:
        return f"CLASSE_DESCONHECIDA: {classe}"
    
    modalidades_permitidas = MODALIDADES_POR_CLASSE[classe]
    if modalidade not in modalidades_permitidas:
        return f"INCONSISTENTE: {classe} + {modalidade}"
    
    return "OK"


df["consistencia_tarifaria"] = df.apply(verificar_consistencia_tarifaria, axis=1)

print("Resultado da verificação de consistência tarifária:")
resultado_consistencia = df["consistencia_tarifaria"].value_counts()
for status, qtd in resultado_consistencia.items():
    print(f"  {status}: {qtd}")

# Mostrar inconsistências
inconsistentes = df[df["consistencia_tarifaria"] != "OK"]
if len(inconsistentes) > 0:
    print(f"\nRegistros inconsistentes ({len(inconsistentes)}):")
    inconsistentes[
        ["cod_consumidor", "nom_consumidor", "cod_classe_consumo",
         "cod_modalidade_tarifaria", "consistencia_tarifaria"]
    ].reset_index(drop=True)

## 3. Detecção de Outliers no Consumo

In [ ]:
def detectar_outliers_iqr(serie: pd.Series, fator: float = 3.0) -> pd.Series:
    """
    Detecta outliers usando o método IQR (Intervalo Interquartílico).
    Com fator=3.0, é menos agressivo que o padrão 1.5 — adequado para
    distribuições assimétricas como consumo de energia.
    """
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - fator * iqr
    limite_superior = q3 + fator * iqr
    return (serie < limite_inferior) | (serie > limite_superior)


# Análise por classe (outliers devem ser detectados dentro de cada classe)
df["outlier_consumo"] = False

for classe in df["cod_classe_consumo"].dropna().unique():
    mask_classe = df["cod_classe_consumo"] == classe
    consumos_classe = df.loc[mask_classe, "vlr_consumo_medio_kwh"].dropna()
    
    if len(consumos_classe) > 3:  # precisa de pelo menos 4 valores
        outliers_mask = detectar_outliers_iqr(consumos_classe)
        df.loc[consumos_classe[outliers_mask].index, "outlier_consumo"] = True

outliers = df[df["outlier_consumo"]]
print(f"Registros com consumo outlier detectado: {len(outliers)}")

if len(outliers) > 0:
    print()
    outliers[
        ["cod_consumidor", "nom_consumidor", "cod_classe_consumo",
         "vlr_consumo_medio_kwh"]
    ].reset_index(drop=True)

In [ ]:
# Estatísticas de consumo por classe (para contextualizar os outliers)
stats_consumo = df.groupby("cod_classe_consumo")["vlr_consumo_medio_kwh"].agg(
    n="count",
    media="mean",
    mediana="median",
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
    maximo="max"
).round(1)

print("Estatísticas de consumo por classe:")
stats_consumo

## 4. Unicidade — Verificação de Duplicatas

In [ ]:
def analisar_unicidade(df: pd.DataFrame, chaves: list) -> dict:
    """
    Analisa a unicidade de conjuntos de chaves no DataFrame.
    """
    resultados = {}
    
    for chave in chaves:
        col = chave if isinstance(chave, list) else [chave]
        duplicatas = df[df.duplicated(subset=col, keep=False)]
        resultados[str(chave)] = {
            "total_registros": len(df),
            "registros_duplicados": len(duplicatas),
            "pct_duplicados": round(len(duplicatas) / len(df) * 100, 1)
        }
    
    return resultados


chaves_analisar = [
    "cod_consumidor",
    "num_medidor",
    ["nom_consumidor", "nom_municipio"]
]

resultado_unicidade = analisar_unicidade(df, chaves_analisar)

print("Análise de Unicidade:")
for chave, resultado in resultado_unicidade.items():
    status = "OK" if resultado["registros_duplicados"] == 0 else "ATENÇÃO"
    print(f"  Chave: {chave}")
    print(f"    Registros duplicados: {resultado['registros_duplicados']} ({resultado['pct_duplicados']}%) — {status}")

## 5. Relatório Consolidado de Qualidade

In [ ]:
# Verificar CPF/CNPJ (usando função do notebook anterior)
def classificar_documento(doc) -> str:
    if pd.isna(doc) or str(doc).strip() in ["", "NAO_INFORMADO", "nan"]:
        return "NAO_INFORMADO"
    digitos = re.sub(r'\D', '', str(doc))
    if len(digitos) == 11:
        return "CPF_INVALIDO_SEQUENCIA" if digitos == digitos[0] * 11 else "CPF"
    elif len(digitos) == 14:
        return "CNPJ_INVALIDO_SEQUENCIA" if digitos == digitos[0] * 14 else "CNPJ"
    return f"FORMATO_INVALIDO_{len(digitos)}_DIGITOS"

df["tipo_documento"] = df["cpf_cnpj"].apply(classificar_documento)

# Compilar problemas por registro
def listar_problemas(row) -> list:
    problemas = []
    
    if row["tipo_documento"] not in ["CPF", "CNPJ"]:
        problemas.append(f"DOC:{row['tipo_documento']}")
    
    if pd.isna(row["num_medidor"]):
        problemas.append("SEM_MEDIDOR")
    
    if pd.isna(row["dat_ultima_leitura"]):
        problemas.append("SEM_DATA_LEITURA")
    
    if pd.notna(row.get("consistencia_tarifaria")) and row.get("consistencia_tarifaria") != "OK":
        problemas.append("INCONSISTENCIA_TARIFARIA")
    
    if row.get("outlier_consumo", False):
        problemas.append("OUTLIER_CONSUMO")
    
    cep = str(row.get("num_cep", ""))
    if not re.match(r'^\d{5}-?\d{3}$', cep):
        problemas.append("CEP_INVALIDO")
    
    return problemas

df["problemas"] = df.apply(listar_problemas, axis=1)
df["qtd_problemas"] = df["problemas"].apply(len)

# Resumo geral
print("=" * 60)
print("RELATÓRIO DE QUALIDADE — CADASTRO DE CONSUMIDORES")
print("=" * 60)
print(f"\nTotal de registros analisados: {len(df)}")
print(f"Registros sem problemas: {(df['qtd_problemas'] == 0).sum()} ({(df['qtd_problemas'] == 0).sum()/len(df)*100:.1f}%)")
print(f"Registros com 1+ problemas: {(df['qtd_problemas'] > 0).sum()} ({(df['qtd_problemas'] > 0).sum()/len(df)*100:.1f}%)")
print(f"\nScore médio de completude: {df['score_completude'].mean():.1%}")

# Contagem por tipo de problema
from collections import Counter
todos_problemas = [p for lista in df["problemas"] for p in lista]
contagem_problemas = Counter(todos_problemas)

print("\nProblemas mais frequentes:")
for problema, qtd in contagem_problemas.most_common():
    pct = qtd / len(df) * 100
    print(f"  {problema:<35}: {qtd:>3} registros ({pct:.1f}%)")

In [ ]:
# Registros prioritários para correção (com mais problemas)
print("\nRegistros prioritários para correção:")
prioritarios = (
    df[df["qtd_problemas"] > 0]
    .sort_values("qtd_problemas", ascending=False)
    [["cod_consumidor", "nom_consumidor", "qtd_problemas", "problemas"]]
    .head(15)
    .reset_index(drop=True)
)
prioritarios.index = prioritarios.index + 1
prioritarios

In [ ]:
# Exportar relatório para Excel
with pd.ExcelWriter("dados/relatorio_qualidade.xlsx", engine="openpyxl") as writer:
    # Aba 1: Todos os registros com scores
    df[
        ["cod_consumidor", "nom_consumidor", "cod_classe_consumo",
         "score_completude", "qtd_problemas", "problemas"]
    ].to_excel(writer, sheet_name="Todos_Registros", index=False)
    
    # Aba 2: Registros com problemas
    df[df["qtd_problemas"] > 0][
        ["cod_consumidor", "nom_consumidor", "cod_classe_consumo",
         "cpf_cnpj", "num_medidor", "qtd_problemas", "problemas"]
    ].to_excel(writer, sheet_name="Com_Problemas", index=False)
    
    # Aba 3: Completude por coluna
    analisar_completude(df).to_excel(writer, sheet_name="Completude", index=False)

print("Relatório exportado: dados/relatorio_qualidade.xlsx")